# Data Preparation: Geocoding Pickup Addresses

This notebook covers the first stage of the logistics feasibility analysis: transforming a raw list of seller pickup addresses into a geocoded dataset with latitude/longitude coordinates.

**The problem:** The carrier received 270 seller addresses from the e-commerce client with no geographic coordinates. Before any route clustering or fleet dimensioning can happen, every address must be mapped to a point on the globe.

**Strategy:** Two-pass geocoding pipeline:
1. **Pass 1 — ViaCEP + Nominatim:** Use the Brazilian postal code (CEP) to resolve a clean address string, then geocode it via Nominatim (OpenStreetMap).
2. **Pass 2 — Retry failures:** Addresses that returned `NaN` coordinates on the first pass are retried with the same pipeline (often succeeds after the Nominatim cache warms up or after rate-limit delays pass).

> **Note on API limits:** Nominatim enforces a 1 req/sec rate limit for anonymous users. The pipeline includes `time.sleep()` calls to comply. Expect the full geocoding run to take 10–15 minutes for 270 addresses. Notebooks 02 and 03 can run offline using the pre-geocoded `enderecos_com_coordenadas.csv` output.

---

## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import time
import requests
import warnings
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

## 2. Load Raw Data

The source file is a prospect sheet provided by the e-commerce client listing their active sellers in the São Paulo metro area. Each row is one seller pickup point.

In [ ]:
file_path = '../data/raw/TT Prospec.xlsx'
nome_sheet = 'Planilha3'

df = pd.read_excel(file_path, sheet_name=nome_sheet)
print(f"Rows: {len(df)} | Columns: {df.shape[1]}")
df.head()

### Anonymize client references

Column names and client identifiers are generalized before any further processing.

In [ ]:
df = df.rename(columns={
    'VOLUME TIKTOK PICKUP': 'VOLUME_PICKUP',
    'CLIENTE': 'CLIENT'
})
if 'CLIENT' in df.columns:
    df['CLIENT'] = 'E-commerce Client'

print("Columns:", df.columns.tolist())

## 3. Data Quality Check

In [ ]:
print("=== Missing values ===")
print(df.isnull().sum())
print()
print("=== Data types ===")
print(df.dtypes)
print()
print("=== Top pickup areas ===")
if 'AREA' in df.columns:
    print(df['AREA'].value_counts().head(10))

In [ ]:
if 'VOLUME_PICKUP' in df.columns:
    print(f"Volume stats:")
    print(df['VOLUME_PICKUP'].describe())
    print(f"\nTotal volume: {df['VOLUME_PICKUP'].sum():,} packages")

## 4. Geocoding Pipeline

The pipeline works in two steps:

**Step A — CEP → clean address string (ViaCEP API)**
- The raw `ZIP CODE` column contains Brazilian CEPs (8-digit postal codes).
- [ViaCEP](https://viacep.com.br) returns structured address data (street, neighborhood, city, state) for any valid CEP.
- We assemble: `"Street, City - State"` as the geocoding query string.

**Step B — Address string → coordinates (Nominatim/OpenStreetMap)**
- Nominatim is queried with the clean address string.
- Exponential backoff (2s, 4s, 8s) handles timeouts; the pipeline retries up to 3 times.
- A 2-second sleep between requests respects Nominatim's usage policy.

In [ ]:
def obter_endereco_string_unica(cep):
    """CEP → formatted address string via ViaCEP API"""
    cep = ''.join(filter(str.isdigit, str(cep)))
    if len(cep) != 8:
        return "Erro: CEP inválido."

    url = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("erro"):
            return "Erro: CEP não encontrado."

        logradouro = data.get('logradouro', '')
        localidade = data.get('localidade', '')
        uf = data.get('uf', '')
        return f"{logradouro}, {localidade} - {uf}"

    except requests.exceptions.RequestException as e:
        return f"Erro na requisição: {e}"


def adicionar_endereco_completo(df):
    """Add ENDERECO_COMPLETO column from ZIP CODE via ViaCEP"""
    print("Resolving addresses from CEPs...")
    enderecos = []

    for index, zipcode in df['ZIP CODE'].items():
        endereco = obter_endereco_string_unica(str(zipcode))
        enderecos.append(endereco)
        time.sleep(0.5)

    df['ENDERECO_COMPLETO'] = enderecos
    return df


def geocode_addresses(df):
    """Add LATITUDE/LONGITUDE via Nominatim, with exponential-backoff retry"""
    geolocator = Nominatim(user_agent="logistics_geocoder_v1", timeout=10)
    latitudes, longitudes = [], []

    print("Geocoding addresses...")
    for index, endereco in df['ENDERECO_COMPLETO'].items():
        if str(endereco).startswith("Erro:"):
            latitudes.append(None)
            longitudes.append(None)
            continue

        max_retries, retry_count, success = 3, 0, False
        while retry_count < max_retries and not success:
            try:
                location = geolocator.geocode(endereco, timeout=10)
                latitudes.append(location.latitude if location else None)
                longitudes.append(location.longitude if location else None)
                success = True
            except (GeocoderTimedOut, GeocoderServiceError) as e:
                retry_count += 1
                if retry_count < max_retries:
                    time.sleep(2 ** retry_count)
                else:
                    latitudes.append(None)
                    longitudes.append(None)
            except Exception:
                latitudes.append(None)
                longitudes.append(None)
                success = True

        time.sleep(2)

    df['LATITUDE'] = latitudes
    df['LONGITUDE'] = longitudes
    return df


def processar_geocodificacao_completa(df):
    """Full geocoding pipeline: CEP → address string → coordinates"""
    print("=== GEOCODING PIPELINE ===")
    df_com_endereco = adicionar_endereco_completo(df.copy())

    validos = df_com_endereco['ENDERECO_COMPLETO'].apply(
        lambda x: not str(x).startswith("Erro:")
    ).sum()
    print(f"Valid CEPs resolved: {validos}/{len(df_com_endereco)} ({validos/len(df_com_endereco)*100:.1f}%)")

    df_final = geocode_addresses(df_com_endereco)

    coords_ok = df_final['LATITUDE'].notna().sum()
    print(f"Coordinates obtained: {coords_ok}/{len(df_final)} ({coords_ok/len(df_final)*100:.1f}%)")

    return df_final

## 5. First Pass — Nominatim Geocoding

Running the full pipeline on all 270 addresses. This takes ~10 minutes due to API rate limits.

In [ ]:
df_resultado = processar_geocodificacao_completa(df)

## 6. Second Pass — Retry Failed Addresses

Addresses that returned `NaN` coordinates on the first pass are retried. Transient Nominatim timeouts account for most first-pass failures.

In [ ]:
df_falhas = df_resultado[df_resultado['LATITUDE'].isna()].copy()
print(f"Addresses to retry: {len(df_falhas)}")

if len(df_falhas) > 0:
    df_falhas = df_falhas.drop(columns=['LATITUDE', 'LONGITUDE', 'ENDERECO_COMPLETO'], errors='ignore')
    df_resultado2 = processar_geocodificacao_completa(df_falhas)

    # Merge retry results back into main dataframe
    df_resultado.update(df_resultado2[['LATITUDE', 'LONGITUDE', 'ENDERECO_COMPLETO']])
    print(f"After retry — coordinates obtained: {df_resultado['LATITUDE'].notna().sum()}/{len(df_resultado)}")
else:
    print("No failed addresses to retry.")

## 7. Merge & Save Output

In [ ]:
print("=== Final geocoding summary ===")
total = len(df_resultado)
ok = df_resultado['LATITUDE'].notna().sum()
print(f"Total addresses: {total}")
print(f"Successfully geocoded: {ok} ({ok/total*100:.1f}%)")
print(f"Missing coordinates: {total - ok}")
print()
df_resultado[['ADDRESS', 'CITY', 'ZIP CODE', 'ENDERECO_COMPLETO', 'LATITUDE', 'LONGITUDE']].head(10)

In [ ]:
output_path = '../data/processed/enderecos_com_coordenadas.csv'
df_resultado.to_csv(output_path, index=False)
print(f"Saved: {output_path}")